In [24]:
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install anywidget

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [anywidget]
Note: you may need to restart the kernel to use updated packages.


In [30]:
import os
import math
from datetime import datetime, timezone
from typing import Dict, Any, Tuple
from functools import lru_cache

import numpy as np
import pandas as pd
import requests
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestClassifier

import ipywidgets as widgets
from IPython.display import display, clear_output

# ============================================================
# CONFIGURATION & MICROGRID ELECTRICAL PARAMETERS
# ============================================================
STATION_LAT = float(os.getenv("STATION_LAT", "-78.1586"))
STATION_LON = float(os.getenv("STATION_LON", "16.3768"))
API_TIMEOUT = int(os.getenv("API_TIMEOUT", "8"))

MICROGRID_VOLTAGE_V = 480.0  # Industrial 3-phase standard
BATTERY_CAPACITY_KWH = 12000.0
BATTERY_MAX_CHARGE_KW = 2500.0
BATTERY_MAX_DISCHARGE_KW = 2500.0
BATTERY_MIN_SOC = 15.0
BATTERY_MAX_SOC = 95.0
BATTERY_ROUND_TRIP_EFF = 0.90

DIESEL_MAX_KW = 5000.0
SOLAR_MAX_KW = 4000.0
WIND_MAX_KW = 6000.0
BASE_LOAD_KW = 3200.0

simulation_state = {
    "battery_soc": 72.0,
    "history": pd.DataFrame()
}

# ============================================================
# LOGIC & MODELS (Weather, Operations, AI Dispatch)
# ============================================================

def get_weather() -> Dict[str, Any]:
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": STATION_LAT, "longitude": STATION_LON, 
        "hourly": "temperature_2m,windspeed_10m,wind_gusts_10m,snowfall,snow_depth,visibility,shortwave_radiation",
        "forecast_days": 3, "wind_speed_unit": "ms", "timezone": "UTC"
    }
    try:
        r = requests.get(url, params=params, timeout=API_TIMEOUT)
        r.raise_for_status()
        h = r.json()["hourly"]
        
        rad_list = h.get("shortwave_radiation", [0]*24)
        daylight_hours = sum(1 for rad in rad_list[:24] if rad > 10.0)
        
        # Open-Meteo returns snow_depth in meters; convert to cm with a realistic floor
        raw_snow_depth = h.get("snow_depth", [0.0] * len(h.get("time", [0]*24)))[0]
        snow_depth_val = float(raw_snow_depth) * 100.0 if raw_snow_depth is not None else 0.0
        snow_depth_final = max(35.0, snow_depth_val)  # Persistent polar snowpack floor
        
        # Guaranteed minimum solar radiation floor for ambient polar twilight/glare
        solar_rad_raw = float(rad_list[0]) if rad_list else 0.0
        solar_rad_final = max(85.0, solar_rad_raw)
        
        return {
            "temperature_c": float(h["temperature_2m"][0]),
            "wind_speed_ms": float(h["windspeed_10m"][0]),
            "wind_gust_ms": float(h["wind_gusts_10m"][0]),
            "snowfall_cm": float(h["snowfall"][0]),
            "visibility_km": float(h["visibility"][0]) / 1000.0,
            "solar_radiation_wm2": solar_rad_final,
            "snow_depth_cm": round(snow_depth_final, 1),
            "daylight_hours": max(2, daylight_hours),
            "source": "Open-Meteo Live",
            "hourly": h
        }
    except Exception:
        return {
            "temperature_c": -18.5, "wind_speed_ms": 11.2, "wind_gust_ms": 16.5,
            "snowfall_cm": 1.2, "visibility_km": 4.5, "solar_radiation_wm2": 140.0,
            "snow_depth_cm": 45.0, "daylight_hours": 10, "source": "Dummy Fallback"
        }

@lru_cache(maxsize=1)
def train_blizzard_model():
    rng = np.random.default_rng(42)
    n = 6000
    wind, gust, snowfall, visibility, temp, pressure_drop = [
        rng.uniform(0, 35, n), rng.uniform(0, 35, n), 
        rng.uniform(0, 12, n), rng.uniform(0.1, 15, n), 
        rng.uniform(-45, 3, n), rng.uniform(0, 30, n)
    ]
    risk_score = 0.07 * wind + 0.08 * gust + 0.24 * snowfall + 0.32 * (10 - visibility) + 0.10 * pressure_drop
    y = (risk_score > 4.2).astype(int)
    model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, class_weight="balanced")
    model.fit(np.column_stack([wind, gust, snowfall, visibility, temp, pressure_drop]), y)
    return model

def predict_blizzard_risk(weather: Dict[str, Any]) -> float:
    model = train_blizzard_model()
    if "hourly" not in weather:
        return 0.25
    h = weather["hourly"]
    probs = []
    for i in range(min(24, len(h.get("time", [0]*24)))):
        try:
            X = [[float(h["windspeed_10m"][i]), float(h["wind_gusts_10m"][i]), float(h["snowfall"][i]),
                  max(0.05, float(h["visibility"][i]) / 1000), float(h["temperature_2m"][i]), 5.0]]
            probs.append(float(model.predict_proba(X)[0, 1]))
        except Exception:
            pass
    return max(probs) if probs else 0.25

def calculate_station_loads(weather: Dict[str, Any]) -> Dict[str, float]:
    temp = weather["temperature_c"]
    heating_kw = 1250.0 + max(0.0, (-15.0 - temp) * 45.0)
    water_kw = 320.0
    
    total_load = BASE_LOAD_KW + heating_kw + water_kw
    
    solar_gen = max(200.0, SOLAR_MAX_KW * min(1.0, weather["solar_radiation_wm2"] / 600.0) * max(0.2, 1.0 - weather["snowfall_cm"] * 0.05))
    wind_gen = max(350.0, WIND_MAX_KW * min(1.0, weather["wind_speed_ms"] / 18.0))
    
    return {
        "load_kw": round(total_load, 1),
        "heating_kw": round(heating_kw, 1),
        "water_kw": round(water_kw, 1),
        "solar_kw": round(solar_gen, 1),
        "wind_kw": round(wind_gen, 1)
    }

def dispatch_energy(loads: Dict[str, float], risk_now: float) -> Dict[str, float]:
    load = loads["load_kw"]
    solar = loads["solar_kw"]
    wind = loads["wind_kw"]
    renewable = solar + wind
    deficit = load - renewable
    soc = simulation_state["battery_soc"]
    
    reserve = 35.0 if risk_now >= 0.70 else 20.0
    battery_kw, diesel_kw = 0.0, 0.0
    
    if deficit > 0:
        usable_soc = max(0.0, soc - reserve)
        if usable_soc > 0:
            battery_kw = min(deficit, min(BATTERY_MAX_DISCHARGE_KW, (usable_soc / 100.0) * BATTERY_CAPACITY_KWH))
        remaining = deficit - battery_kw
        if remaining > 0:
            diesel_kw = min(DIESEL_MAX_KW, remaining)
    else:
        excess = -deficit
        available_cap = ((BATTERY_MAX_SOC - soc) / 100.0) * BATTERY_CAPACITY_KWH
        battery_kw = -min(excess, BATTERY_MAX_CHARGE_KW, max(0.0, available_cap))

    if battery_kw > 0:
        soc -= ((battery_kw * (1.0 / 60.0) / BATTERY_ROUND_TRIP_EFF) / BATTERY_CAPACITY_KWH) * 100
    elif battery_kw < 0:
        soc += ((-battery_kw * (1.0 / 60.0) * BATTERY_ROUND_TRIP_EFF) / BATTERY_CAPACITY_KWH) * 100

    soc = float(np.clip(soc, BATTERY_MIN_SOC, BATTERY_MAX_SOC))
    simulation_state["battery_soc"] = soc

    net_shortage = max(0.0, deficit - battery_kw - diesel_kw)
    net_discharge_rate = max(0.1, battery_kw)
    available_kwh = max(0.0, (soc - BATTERY_MIN_SOC) / 100.0 * BATTERY_CAPACITY_KWH)
    depletion_hours = available_kwh / net_discharge_rate if battery_kw > 0 else float('inf')

    return {
        "load_kw": load, "solar_kw": solar, "wind_kw": wind, "diesel_kw": diesel_kw,
        "battery_kw": battery_kw, "battery_soc": soc, "shortage_kw": net_shortage,
        "depletion_hours": depletion_hours
    }

# ============================================================
# VIBRANT DARK-THEME DASHBOARD UI COMPONENTS
# ============================================================

title_widget = widgets.HTML("""
    <div style='background: linear-gradient(135deg, #1e1b4b, #312e81); padding: 22px; border-radius: 12px; color: white; border-bottom: 4px solid #6366f1; box-shadow: 0 10px 15px -3px rgba(0,0,0,0.3);'>
        <h2 style='margin: 0; font-family: sans-serif; font-weight: 700; letter-spacing: 0.05em; color: #a5f3fc;'>❄️ ARCTIC & ANTARCTIC RESEARCH STATION</h2>
        <p style='margin: 6px 0 0 0; color: #cbd5e1; font-size: 13px;'>AI-Powered Resilient Microgrid Energy & Telemetry Management System</p>
    </div>
""")

status_widget = widgets.HTML("""
    <div style='background-color: #0f172a; padding: 12px 18px; border-radius: 8px; color: #38bdf8; font-size: 12px; margin-top: 12px; border: 1px solid #334155; font-weight: 600;'>
        <b>System Status:</b> Synchronized with telemetry feeds. All primary sub-systems operational.
    </div>
""")

def make_card(title, value, subtitle="", accent_color="#38bdf8"):
    sub_html = f"<div style='margin-top: 6px; font-size: 11px; color: #94a3b8; font-weight: 500;'>{subtitle}</div>" if subtitle else ""
    return widgets.HTML(f"""
        <div style='background: linear-gradient(145deg, #1e293b, #0f172a); padding: 16px; border-radius: 10px; text-align: center; border: 1px solid #334155; border-top: 3px solid {accent_color}; box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.2); width: 100%;'>
            <div style='font-size: 11px; text-transform: uppercase; letter-spacing: 0.08em; color: #94a3b8; font-weight: 700;'>{title}</div>
            <div style='font-size: 22px; font-weight: 800; color: {accent_color}; margin-top: 8px;'>{value}</div>
            {sub_html}
        </div>
    """)

# Weather Metric Cards (Vibrant Accent Colors)
w_temp = make_card("Temperature", "-18.5 °C", accent_color="#f43f5e")
w_wind = make_card("Wind Speed", "11.2 m/s", accent_color="#38bdf8")
w_solar_rad = make_card("Solar Radiation", "140 W/m²", accent_color="#fbbf24")
w_snow = make_card("Snow / Depth", "1.2 / 45.0 cm", accent_color="#a855f7")
w_vis = make_card("Visibility / Day", "4.5 km / 10h", accent_color="#2dd4bf")
w_risk = make_card("Blizzard Risk", "25.0%", "AI Forecast", accent_color="#ec4899")

weather_box = widgets.HBox([w_temp, w_wind, w_solar_rad, w_snow, w_vis, w_risk], layout=widgets.Layout(justify_content='space-between', gap='10px'))

# Grid Cards for Power & Currents
card_in_power = make_card("Total Input Power", "4,850 kW", "Generation + Discharge", accent_color="#10b981")
card_gen = make_card("Renewable Gen", "1,950 kW", "Solar + Wind", accent_color="#34d399")
card_diesel = make_card("Diesel Output", "1,200 kW", "Backup Generation", accent_color="#f59e0b")

card_in_curr = make_card("Input Current", "10,104 A", f"@ {MICROGRID_VOLTAGE_V}V System", accent_color="#10b981")
card_consumption = make_card("Total Consumption", "4,770 kW", "Load + Charging", accent_color="#60a5fa")
card_out_curr = make_card("Consumption Current", "9,937 A", f"@ {MICROGRID_VOLTAGE_V}V System", accent_color="#60a5fa")

output_box1 = widgets.HBox([card_in_power, card_gen, card_diesel], layout=widgets.Layout(justify_content='space-between', gap='10px'))
output_box2 = widgets.HBox([card_in_curr, card_consumption, card_out_curr], layout=widgets.Layout(justify_content='space-between', gap='10px', margin='10px 0 0 0'))

strategy_widget = widgets.HTML("""
    <div style='background: linear-gradient(135deg, #1e293b, #0f172a); padding: 18px; border-radius: 10px; border-left: 5px solid #6366f1; border: 1px solid #334155; color: #f8fafc; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);'>
        <h4 style='margin: 0 0 8px 0; color: #a5f3fc; font-family: sans-serif;'>🧠 AI Recommendations & Critical-Load Protection</h4>
        <p style='margin: 0; color: #cbd5e1; font-size: 13px; line-height: 1.5;'>Analyzing real-time polar telemetry and automated microgrid load distribution...</p>
    </div>
""")

chart_output = widgets.Output()
refresh_btn = widgets.Button(description="🔄 Refresh Telemetry & Dispatch", button_style='primary', layout=widgets.Layout(width='280px', height='45px'))

def update_dashboard(b=None):
    weather = get_weather()
    loads = calculate_station_loads(weather)
    max_risk = predict_blizzard_risk(weather)
    result = dispatch_energy(loads, max_risk)
    
    # Update Weather Cards with Dynamic Live Data
    w_temp.value = f"<div style='font-size:11px;color:#94a3b8;font-weight:700;'>TEMPERATURE</div><div style='font-size:20px;font-weight:800;color:#f43f5e;margin-top:8px;'>{weather['temperature_c']:.1f} °C</div>"
    w_wind.value = f"<div style='font-size:11px;color:#94a3b8;font-weight:700;'>WIND SPEED</div><div style='font-size:20px;font-weight:800;color:#38bdf8;margin-top:8px;'>{weather['wind_speed_ms']:.1f} m/s</div>"
    w_solar_rad.value = f"<div style='font-size:11px;color:#94a3b8;font-weight:700;'>SOLAR RADIATION</div><div style='font-size:20px;font-weight:800;color:#fbbf24;margin-top:8px;'>{weather['solar_radiation_wm2']:.1f} W/m²</div>"
    w_snow.value = f"<div style='font-size:11px;color:#94a3b8;font-weight:700;'>SNOW / DEPTH</div><div style='font-size:17px;font-weight:800;color:#a855f7;margin-top:8px;'>{weather['snowfall_cm']:.1f} / {weather['snow_depth_cm']:.1f} cm</div>"
    w_vis.value = f"<div style='font-size:11px;color:#94a3b8;font-weight:700;'>VISIBILITY / DAY</div><div style='font-size:17px;font-weight:800;color:#2dd4bf;margin-top:8px;'>{weather['visibility_km']:.1f} km / {weather['daylight_hours']}h</div>"
    w_risk.value = f"<div style='font-size:11px;color:#94a3b8;font-weight:700;'>BLIZZARD RISK</div><div style='font-size:20px;font-weight:800;color:#ec4899;margin-top:8px;'>{max_risk * 100:.1f}%</div><div style='font-size:10px;color:#94a3b8;margin-top:2px;'>AI Forecast</div>"

    # Electrical Calculations
    renew_kw = result["solar_kw"] + result["wind_kw"]
    batt_discharge = max(0.0, result["battery_kw"])
    batt_charge = max(0.0, -result["battery_kw"])
    
    total_input_kw = renew_kw + result["diesel_kw"] + batt_discharge
    total_consumption_kw = result["load_kw"] + batt_charge
    
    input_curr_a = (total_input_kw * 1000.0) / MICROGRID_VOLTAGE_V
    consumption_curr_a = (total_consumption_kw * 1000.0) / MICROGRID_VOLTAGE_V

    # Update Output & Current Grid Cards
    card_in_power.value = f"<div style='font-size:11px;color:#94a3b8;font-weight:700;'>TOTAL INPUT POWER</div><div style='font-size:20px;font-weight:800;color:#10b981;margin-top:8px;'>{total_input_kw:,.1f} kW</div><div style='font-size:11px;color:#94a3b8;margin-top:2px;'>Generation + Discharge</div>"
    card_gen.value = f"<div style='font-size:11px;color:#94a3b8;font-weight:700;'>RENEWABLE GEN</div><div style='font-size:20px;font-weight:800;color:#34d399;margin-top:8px;'>{renew_kw:,.1f} kW</div><div style='font-size:11px;color:#94a3b8;margin-top:2px;'>Solar + Wind</div>"
    card_diesel.value = f"<div style='font-size:11px;color:#94a3b8;font-weight:700;'>DIESEL OUTPUT</div><div style='font-size:20px;font-weight:800;color:#f59e0b;margin-top:8px;'>{result['diesel_kw']:,.1f} kW</div><div style='font-size:11px;color:#94a3b8;margin-top:2px;'>Backup Generation</div>"
    
    card_in_curr.value = f"<div style='font-size:11px;color:#94a3b8;font-weight:700;'>INPUT CURRENT</div><div style='font-size:20px;font-weight:800;color:#10b981;margin-top:8px;'>{input_curr_a:,.1f} A</div><div style='font-size:11px;color:#94a3b8;margin-top:2px;'>@ {MICROGRID_VOLTAGE_V}V System</div>"
    card_consumption.value = f"<div style='font-size:11px;color:#94a3b8;font-weight:700;'>TOTAL CONSUMPTION</div><div style='font-size:20px;font-weight:800;color:#60a5fa;margin-top:8px;'>{total_consumption_kw:,.1f} kW</div><div style='font-size:11px;color:#94a3b8;margin-top:2px;'>Load + Charging</div>"
    card_out_curr.value = f"<div style='font-size:11px;color:#94a3b8;font-weight:700;'>CONSUMPTION CURRENT</div><div style='font-size:20px;font-weight:800;color:#60a5fa;margin-top:8px;'>{consumption_curr_a:,.1f} A</div><div style='font-size:11px;color:#94a3b8;margin-top:2px;'>@ {MICROGRID_VOLTAGE_V}V System</div>"

    # AI Recommendations Text
    depletion_str = f"{result['depletion_hours']:.1f} hours" if result['depletion_hours'] != float('inf') else "Stable (>48h)"
    shortage_str = f"{result['shortage_kw']:.1f} kW" if result['shortage_kw'] > 0 else "None (Balanced)"
    
    if max_risk >= 0.70:
        strat = f"🔴 <b style='color:#ef4444;'>CRITICAL WARNING:</b> Severe blizzard risk predicted! <br>• <b>Battery Depletion Buffer:</b> {depletion_str} remaining. <br>• <b>Energy Deficit:</b> {shortage_str}. <br>• <b>Action:</b> Curtail secondary heating zones, postpone non-essential research workloads, and engage backup diesel units."
    elif max_risk >= 0.40:
        strat = f"🟠 <b style='color:#f59e0b;'>MODERATE ALERT:</b> Elevated atmospheric volatility. <br>• <b>Battery Depletion Buffer:</b> {depletion_str}. <br>• <b>Action:</b> Store excess clean energy reserves, maintain generator standby readiness."
    else:
        strat = f"🟢 <b style='color:#10b981;'>NORMAL OPERATION:</b> Stable polar meteorological conditions. <br>• <b>Battery Depletion Buffer:</b> {depletion_str}. <br>• <b>Action:</b> Active renewables and thermal microgrid systems operating at peak efficiency."

    strategy_widget.value = f"""
        <div style='background: linear-gradient(135deg, #1e293b, #0f172a); padding: 18px; border-radius: 10px; border-left: 5px solid #6366f1; border: 1px solid #334155; color: #f8fafc; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);'>
            <h4 style='margin: 0 0 8px 0; color: #a5f3fc; font-family: sans-serif;'>🧠 AI Recommendations & Critical-Load Protection</h4>
            <p style='margin: 0; color: #cbd5e1; font-size: 13px; line-height: 1.5;'>{strat}</p>
        </div>
    """

    # Interactive Graph with Vibrant Dark-Theme Styling
    with chart_output:
        clear_output(wait=True)
        labels = ["Solar", "Wind", "Diesel", "Battery Net", "Station Load", "Heating Load", "Water Treatment"]
        values = [result["solar_kw"], result["wind_kw"], result["diesel_kw"], result["battery_kw"], result["load_kw"], loads["heating_kw"], loads["water_kw"]]
        
        fig = go.Figure(go.Bar(
            x=labels, y=values, 
            text=[f"{v:,.0f} kW" for v in values], 
            textposition="auto",
            marker_color=['#fbbf24', '#38bdf8', '#f59e0b', '#34d399', '#60a5fa', '#f43f5e', '#2dd4bf']
        ))
        fig.update_layout(
            title="Microgrid Generation vs Consumption & Specialized Loads",
            height=380,
            yaxis_title="Power (kW)",
            paper_bgcolor="#0f172a",
            plot_bgcolor="#1e293b",
            font=dict(color="#f8fafc", family="sans-serif"),
            margin=dict(l=20, r=20, t=40, b=20),
            xaxis=dict(gridcolor="#334155"),
            yaxis=dict(gridcolor="#334155")
        )
        fig.show()

    status_widget.value = f"""
        <div style='background-color: #0f172a; padding: 12px 18px; border-radius: 8px; color: #38bdf8; font-size: 12px; margin-top: 12px; border: 1px solid #334155; font-weight: 600;'>
            <b>System Status:</b> Refreshed successfully at {datetime.now().strftime('%H:%M:%S')} | <b>Battery SOC:</b> {result['battery_soc']:.1f}%
        </div>
    """

refresh_btn.on_click(update_dashboard)

# Assemble Professional Dashboard Layout Container
dashboard_layout = widgets.VBox([
    title_widget,
    status_widget,
    widgets.HTML("<hr style='border-color: #334155; margin: 15px 0;'>"),
    widgets.HTML("<h4 style='color: #a5f3fc; font-family: sans-serif; margin-bottom: 10px; font-weight: 600;'>🌨️ 1. Live Polar Weather & Environmental Factors</h4>"),
    weather_box,
    widgets.HTML("<h4 style='color: #a5f3fc; font-family: sans-serif; margin-top: 18px; margin-bottom: 10px; font-weight: 600;'>⚡ 2. Microgrid Operational Input & Consumption Currents</h4>"),
    output_box1,
    output_box2,
    widgets.HTML("<h4 style='color: #a5f3fc; font-family: sans-serif; margin-top: 18px; margin-bottom: 10px; font-weight: 600;'>🧠 3. AI Recommendations & Critical-Load Protection</h4>"),
    strategy_widget,
    widgets.HTML("<h4 style='color: #a5f3fc; font-family: sans-serif; margin-top: 18px; margin-bottom: 10px; font-weight: 600;'>📈 4. Interactive Graphical Representations</h4>"),
    chart_output,
    widgets.HTML("<br>"),
    refresh_btn
], layout=widgets.Layout(background_color='#0f172a', padding='20px', border_radius='12px', box_shadow='0 20px 25px -5px rgba(0,0,0,0.4)'))

update_dashboard()
display(dashboard_layout)